# Task 3 — NOVA Product Knowledge RAG

ChromaDB + **hybrid** (dense + BM25) + **cross-encoder re-ranking** via `rag_module.py`.
Optional: **RAGAS** metrics (install `ragas`, set `GROQ_API_KEY` for judge LLM).

In [ ]:
%pip install -q chromadb sentence-transformers rank-bm25 python-dotenv ragas datasets langchain-openai
import os
from pathlib import Path
import json
from dotenv import load_dotenv

load_dotenv()
ROOT = Path.cwd()
os.chdir(ROOT)

In [ ]:
from rag_module import HybridRAG, load_mock_catalog

persist = ROOT / ".chroma" / "nova_kb_colab"
rag = HybridRAG(persist_dir=str(persist))
n = rag.index_products(load_mock_catalog())
print("Indexed", n, "chunks")

In [ ]:
q = "Does the HydraCalm serum contain niacinamide for sensitive skin?"
for row in rag.retrieve(q, top_k=3):
    print(row["id"], row["rerank_score"])
    print(row["text"][:280], "...\n")

In [ ]:
# RAGAS (optional — needs GROQ_API_KEY for evaluation LLM)
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from datasets import Dataset

ds = Dataset.from_dict({
    "question": [q],
    "answer": ["Yes — HydraCalm lists niacinamide and is labeled fragrance-free for sensitive skin."],
    "contexts": [[row["text"] for row in rag.retrieve(q, top_k=2)]],
})

os.environ["OPENAI_API_KEY"] = os.environ.get("GROQ_API_KEY") or os.environ.get("groq_api_key", "")
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

try:
    out = evaluate(ds, metrics=[faithfulness, answer_relevancy])
    print(out)
    Path("evaluation_report.json").write_text(out.to_pandas().to_json(orient="records", indent=2))
except Exception as e:
    print("RAGAS skipped or failed:", e)